In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time
from collections import deque

In [2]:
def distance(p1, p2):
    return math.sqrt(
        (p2[0] - p1[0]) ** 2 +
        (p2[1] - p1[1]) ** 2
    )


def calculate_angle(a, b, c):
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    c = np.array(c, dtype=np.float32)

    ba = a - b
    bc = c - b

    denominator = np.linalg.norm(ba) * np.linalg.norm(bc)

    if denominator == 0:
        return 0.0

    cosine_angle = np.dot(ba, bc) / denominator
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    return float(np.degrees(np.arccos(cosine_angle)))


def to_pixel(landmark, width, height):
    return (
        int(landmark.x * width),
        int(landmark.y * height)
    )


def clamp(value, minimum, maximum):
    return max(minimum, min(value, maximum))

In [3]:
class OfficePostureMonitor:

    def __init__(
        self,
        neck_threshold=12,
        slouch_threshold=1.20,
        close_threshold=1.25,
        calibration_frames=90,
        smoothing_window=10
    ):

        self.mp_pose = mp.solutions.pose
        self.mp_face_mesh = mp.solutions.face_mesh

        self.pose = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        self.neck_threshold = neck_threshold
        self.slouch_threshold = slouch_threshold
        self.close_threshold = close_threshold

        self.calibration_frames_required = calibration_frames
        self.calibration_count = 0
        self.calibrated = False

        self.calibration_neck = []
        self.calibration_slouch = []
        self.calibration_face_width = []

        self.baseline = {
            "neck_angle": 0.0,
            "slouch_ratio": 0.0,
            "face_width": 0.0
        }

        self.neck_history = deque(maxlen=smoothing_window)
        self.slouch_history = deque(maxlen=smoothing_window)
        self.face_history = deque(maxlen=smoothing_window)

        self.last_metrics = {
            "neck_angle": 0.0,
            "slouch_ratio": 0.0,
            "face_width": 0.0,
            "posture_score": 0,
            "status": "CALIBRATING",
            "alerts": []
        }

    def reset_calibration(self):

        self.calibration_count = 0
        self.calibrated = False

        self.calibration_neck.clear()
        self.calibration_slouch.clear()
        self.calibration_face_width.clear()

        self.neck_history.clear()
        self.slouch_history.clear()
        self.face_history.clear()

    def smooth(self, history, value):

        history.append(value)

        if len(history) == 0:
            return value

        return float(np.mean(history))

    def update_calibration(
        self,
        neck_angle,
        slouch_ratio,
        face_width
    ):

        self.calibration_neck.append(neck_angle)
        self.calibration_slouch.append(slouch_ratio)
        self.calibration_face_width.append(face_width)

        self.calibration_count += 1

        if self.calibration_count >= self.calibration_frames_required:

            self.baseline["neck_angle"] = float(
                np.median(self.calibration_neck)
            )

            self.baseline["slouch_ratio"] = float(
                np.median(self.calibration_slouch)
            )

            self.baseline["face_width"] = float(
                np.median(self.calibration_face_width)
            )

            self.calibrated = True

    def calculate_posture_score(
        self,
        neck_difference,
        slouch_difference,
        face_difference
    ):

        neck_penalty = clamp(
            (neck_difference / max(self.neck_threshold, 1)) * 35,
            0,
            35
        )

        slouch_penalty = clamp(
            (slouch_difference /
             max(self.baseline["slouch_ratio"] * 0.20, 0.01)) * 35,
            0,
            35
        )

        distance_penalty = clamp(
            (face_difference /
             max(self.baseline["face_width"] * 0.25, 1)) * 30,
            0,
            30
        )

        score = 100 - (
            neck_penalty +
            slouch_penalty +
            distance_penalty
        )

        return int(clamp(score, 0, 100))

    def process(self, frame):

        frame = cv2.flip(frame, 1)

        height, width = frame.shape[:2]

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        pose_results = self.pose.process(rgb)
        face_results = self.face_mesh.process(rgb)

        neck_angle = None
        slouch_ratio = None
        face_width = None

        alerts = []

        # ---------------------------------
        # Pose analysis
        # ---------------------------------

        if pose_results.pose_landmarks:

            landmarks = pose_results.pose_landmarks.landmark

            left_shoulder = to_pixel(
                landmarks[
                    self.mp_pose.PoseLandmark.LEFT_SHOULDER.value
                ],
                width,
                height
            )

            right_shoulder = to_pixel(
                landmarks[
                    self.mp_pose.PoseLandmark.RIGHT_SHOULDER.value
                ],
                width,
                height
            )

            left_ear = to_pixel(
                landmarks[
                    self.mp_pose.PoseLandmark.LEFT_EAR.value
                ],
                width,
                height
            )

            right_ear = to_pixel(
                landmarks[
                    self.mp_pose.PoseLandmark.RIGHT_EAR.value
                ],
                width,
                height
            )

            shoulder_center = (
                (left_shoulder[0] + right_shoulder[0]) // 2,
                (left_shoulder[1] + right_shoulder[1]) // 2
            )

            ear_center = (
                (left_ear[0] + right_ear[0]) // 2,
                (left_ear[1] + right_ear[1]) // 2
            )

            vertical_reference = (
                shoulder_center[0],
                shoulder_center[1] - 100
            )

            neck_angle = calculate_angle(
                ear_center,
                shoulder_center,
                vertical_reference
            )

            shoulder_width = distance(
                left_shoulder,
                right_shoulder
            )

            neck_to_shoulder = distance(
                ear_center,
                shoulder_center
            )

            if shoulder_width > 0:
                slouch_ratio = (
                    neck_to_shoulder /
                    shoulder_width
                )
            else:
                slouch_ratio = 0.0

            neck_angle = self.smooth(
                self.neck_history,
                neck_angle
            )

            slouch_ratio = self.smooth(
                self.slouch_history,
                slouch_ratio
            )

            # Draw posture geometry
            cv2.line(
                frame,
                left_shoulder,
                right_shoulder,
                (0, 255, 150),
                3
            )

            cv2.line(
                frame,
                shoulder_center,
                ear_center,
                (0, 255, 150),
                3
            )

            for point in [
                left_shoulder,
                right_shoulder,
                shoulder_center,
                ear_center
            ]:
                cv2.circle(
                    frame,
                    point,
                    5,
                    (0, 255, 150),
                    -1
                )

        # ---------------------------------
        # Face analysis
        # ---------------------------------

        if face_results.multi_face_landmarks:

            face_landmarks = (
                face_results
                .multi_face_landmarks[0]
                .landmark
            )

            left_face = to_pixel(
                face_landmarks[234],
                width,
                height
            )

            right_face = to_pixel(
                face_landmarks[454],
                width,
                height
            )

            face_width = distance(
                left_face,
                right_face
            )

            face_width = self.smooth(
                self.face_history,
                face_width
            )

            cv2.line(
                frame,
                left_face,
                right_face,
                (255, 180, 0),
                2
            )

            cv2.circle(
                frame,
                left_face,
                4,
                (255, 180, 0),
                -1
            )

            cv2.circle(
                frame,
                right_face,
                4,
                (255, 180, 0),
                -1
            )

        # ---------------------------------
        # Calibration
        # ---------------------------------

        valid_measurements = (
            neck_angle is not None and
            slouch_ratio is not None and
            face_width is not None
        )

        if not self.calibrated and valid_measurements:

            self.update_calibration(
                neck_angle,
                slouch_ratio,
                face_width
            )

        # ---------------------------------
        # Posture evaluation
        # ---------------------------------

        neck_difference = 0.0
        slouch_difference = 0.0
        face_difference = 0.0

        if self.calibrated and valid_measurements:

            neck_difference = abs(
                neck_angle -
                self.baseline["neck_angle"]
            )

            slouch_difference = max(
                0,
                slouch_ratio -
                self.baseline["slouch_ratio"]
            )

            face_difference = max(
                0,
                face_width -
                self.baseline["face_width"]
            )

            if neck_difference > self.neck_threshold:
                alerts.append("NECK BENDING")

            if slouch_ratio > (
                self.baseline["slouch_ratio"] *
                self.slouch_threshold
            ):
                alerts.append("SLOUCHING")

            if face_width > (
                self.baseline["face_width"] *
                self.close_threshold
            ):
                alerts.append("TOO CLOSE TO SCREEN")

        # ---------------------------------
        # Status and score
        # ---------------------------------

        if not self.calibrated:

            status = "CALIBRATING"
            posture_score = 0

        elif not valid_measurements:

            status = "PERSON NOT DETECTED"
            posture_score = 0

        elif len(alerts) == 0:

            status = "GOOD POSTURE"

            posture_score = self.calculate_posture_score(
                neck_difference,
                slouch_difference,
                face_difference
            )

        else:

            status = "POSTURE ALERT"

            posture_score = self.calculate_posture_score(
                neck_difference,
                slouch_difference,
                face_difference
            )

        # ---------------------------------
        # Save metrics
        # ---------------------------------

        self.last_metrics = {
            "neck_angle": round(neck_angle or 0, 2),
            "slouch_ratio": round(slouch_ratio or 0, 3),
            "face_width": round(face_width or 0, 1),
            "posture_score": posture_score,
            "status": status,
            "alerts": alerts
        }

        # ---------------------------------
        # Draw dashboard
        # ---------------------------------

        self.draw_dashboard(
            frame,
            neck_difference,
            alerts
        )

        return frame, self.last_metrics

    def draw_dashboard(
        self,
        frame,
        neck_difference,
        alerts
    ):

        height, width = frame.shape[:2]

        overlay = frame.copy()

        cv2.rectangle(
            overlay,
            (20, 20),
            (430, 210),
            (10, 18, 30),
            -1
        )

        cv2.addWeighted(
            overlay,
            0.85,
            frame,
            0.15,
            0,
            frame
        )

        status = self.last_metrics["status"]

        if status == "GOOD POSTURE":
            status_color = (0, 220, 120)

        elif status == "CALIBRATING":
            status_color = (0, 190, 255)

        else:
            status_color = (0, 0, 255)

        cv2.putText(
            frame,
            "OFFICE POSTURE MONITOR",
            (40, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Status: {status}",
            (40, 85),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            status_color,
            2
        )

        if not self.calibrated:

            progress = int(
                100 *
                self.calibration_count /
                self.calibration_frames_required
            )

            progress = clamp(progress, 0, 100)

            cv2.putText(
                frame,
                f"Calibration: {progress}%",
                (40, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1
            )

        else:

            cv2.putText(
                frame,
                f"Neck Angle: {self.last_metrics['neck_angle']:.1f} deg",
                (40, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                1
            )

            cv2.putText(
                frame,
                f"Slouch Ratio: {self.last_metrics['slouch_ratio']:.3f}",
                (40, 150),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                1
            )

            cv2.putText(
                frame,
                f"Posture Score: {self.last_metrics['posture_score']}/100",
                (40, 180),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                1
            )

        # Alerts
        y = 250

        for alert in alerts:

            cv2.rectangle(
                frame,
                (20, y - 28),
                (350, y + 8),
                (20, 20, 80),
                -1
            )

            cv2.putText(
                frame,
                f"ALERT: {alert}",
                (35, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 80, 255),
                2
            )

            y += 45

    def release(self):

        self.pose.close()
        self.face_mesh.close()

In [4]:
monitor = OfficePostureMonitor(
    neck_threshold=12,
    slouch_threshold=1.20,
    close_threshold=1.25,
    calibration_frames=90,
    smoothing_window=10
)

print("Posture monitor created successfully.")

Posture monitor created successfully.


In [5]:
cap = cv2.VideoCapture(0)

# Optional camera settings
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

if not cap.isOpened():
    raise RuntimeError(
        "Unable to open webcam. "
        "Check your camera permissions or camera index."
    )

print("Starting Office Posture Monitor...")
print("Sit correctly during calibration.")
print("Press Q to quit.")
print("Press R to recalibrate.")
print("Press S to display current metrics.")

previous_alert_time = 0
alert_cooldown = 2.0

try:

    while True:

        success, frame = cap.read()

        if not success:
            print("Unable to read camera frame.")
            break

        processed_frame, metrics = monitor.process(frame)

        cv2.imshow(
            "Office Posture Monitor",
            processed_frame
        )

        # Optional terminal alert
        if metrics["alerts"]:

            current_time = time.time()

            if (
                current_time -
                previous_alert_time
                > alert_cooldown
            ):

                print(
                    "POSTURE ALERT:",
                    ", ".join(metrics["alerts"])
                )

                previous_alert_time = current_time

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

        elif key == ord("r"):

            monitor.reset_calibration()

            print(
                "Calibration reset. "
                "Sit in your correct posture."
            )

        elif key == ord("s"):

            print("\nCurrent Metrics")
            print("-" * 40)

            for name, value in metrics.items():
                print(f"{name}: {value}")

finally:

    cap.release()
    monitor.release()

    cv2.destroyAllWindows()

    print("Office Posture Monitor stopped.")

Starting Office Posture Monitor...
Sit correctly during calibration.
Press Q to quit.
Press R to recalibrate.
Press S to display current metrics.
POSTURE ALERT: TOO CLOSE TO SCREEN
POSTURE ALERT: NECK BENDING
POSTURE ALERT: NECK BENDING, TOO CLOSE TO SCREEN
POSTURE ALERT: NECK BENDING, TOO CLOSE TO SCREEN
POSTURE ALERT: NECK BENDING
POSTURE ALERT: NECK BENDING
POSTURE ALERT: NECK BENDING, TOO CLOSE TO SCREEN
POSTURE ALERT: NECK BENDING
POSTURE ALERT: NECK BENDING, TOO CLOSE TO SCREEN
POSTURE ALERT: TOO CLOSE TO SCREEN
POSTURE ALERT: TOO CLOSE TO SCREEN
POSTURE ALERT: NECK BENDING, TOO CLOSE TO SCREEN
POSTURE ALERT: NECK BENDING
POSTURE ALERT: NECK BENDING
POSTURE ALERT: NECK BENDING
POSTURE ALERT: NECK BENDING, TOO CLOSE TO SCREEN
POSTURE ALERT: NECK BENDING
Office Posture Monitor stopped.


In [6]:
# Record the posture monitoring output
# The processed video will be saved in the current folder.

import os
import cv2

# GitHub link displayed in the bottom-right corner of the recorded video
github_link = "github.com/ajanthadevi2012"

# Output file saved in the same/current working folder
output_path = "posture_output.mp4"

video_monitor = OfficePostureMonitor(calibration_frames=60)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Unable to open webcam. Check camera permissions or camera index.")

# Read camera properties
fps = cap.get(cv2.CAP_PROP_FPS)
if fps is None or fps <= 1:
    fps = 30.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print("Recording Office Posture Monitor...")
print(f"GitHub link on video: {github_link}")
print(f"Output will be saved to: {os.path.abspath(output_path)}")
print("Press Q to stop recording.")

while True:
    success, frame = cap.read()

    if not success:
        print("Unable to read frame. Stopping recording.")
        break

    # Process posture
    output_frame, metrics = video_monitor.process(frame)

    # Add GitHub link in the bottom-right corner
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.55
    thickness = 1
    margin = 12

    (text_width, text_height), baseline = cv2.getTextSize(
        github_link, font, font_scale, thickness
    )

    x = width - text_width - margin
    y = height - margin

    # Background rectangle for better visibility
    cv2.rectangle(
        output_frame,
        (x - 8, y - text_height - 8),
        (width, height),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        output_frame,
        github_link,
        (x, y - baseline),
        font,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )

    # Save the processed frame
    writer.write(output_frame)

    # Display live recording
    cv2.imshow("Office Posture Monitor - Recording", output_frame)

    # Press Q to stop recording
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Release resources
cap.release()
writer.release()
video_monitor.release()
cv2.destroyAllWindows()

print("\nRecording completed successfully.")
print(f"Output video saved in the current folder: {os.path.abspath(output_path)}")


Recording Office Posture Monitor...
GitHub link on video: github.com/ajanthadevi2012
Output will be saved to: C:\Users\ajant\Downloads\ProgNxt\Project - Office Posture Monitoring\posture_output.mp4
Press Q to stop recording.

Recording completed successfully.
Output video saved in the current folder: C:\Users\ajant\Downloads\ProgNxt\Project - Office Posture Monitoring\posture_output.mp4
